# Experiment 1: the joint $N,\lambda$ limit on a fixed bandwidth interval

This notebook isolates the spectral asymptotics from the separate sample-size limit. It fixes the data sample size, the underlying distribution, and the bandwidth interval, while varying only the bandwidth-grid size $N$ and the similarity localization $\lambda_N$.

One sample from $N(0,1)$ is drawn once and held fixed throughout. Every $(N,\lambda_N)$ configuration is evaluated on this same sample, so the notebook isolates the grid and localization limits from sample-to-sample variation.

We study the regimes $\lambda_N=N^\alpha$, where $\alpha\in\{0,1/2,1,3/2,2,3\}$.


L² inner products use direct frequency quadrature (`frequency_trapezoid_v1`) with the shared production settings **Omega = 450 and K = 1536 intervals (1537 nodes)**. The compatibility values `half_width = pi*K/Omega` and `spatial_points = 2*K` specify this frequency grid; the L² helper does not reconstruct spatial densities. Configuration identities include the frequency grid, so checkpoints from other grids are not reused. Rerun setup after updating the helpers.

The final section incorporates the r=1 and r=13/14 grid experiments.
These and the repeated-sample sections address the chapter TODOs beside
Figure `fig:N_lambda_joint_limit_for_different_alpha`.

Reusable functions: [`joint_experiments.py`](../kde_tools/joint_experiments.py).
Experiment settings and calls are below. Every displayed figure is saved automatically as a PDF.


In [ ]:
from _setup import setup_notebook
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
import warnings
from IPython.display import display
from experiments.joint_limits import run_experiment as grid_experiment
from kde_tools.bandwidths import bandwidth_grid_linear_in_psi_beta, nearest_bandwidth_grid_index
from kde_tools.energy_minimization import EnergySearchConfig, minimize_bandwidth_energy
from kde_tools.fourier.similarities import (
    DEFAULT_FOURIER_GRID_TAG,
    DEFAULT_FREQUENCY_CUTOFF,
    DEFAULT_FREQUENCY_INTERVALS,
    FOURIER_L2_METHOD_VERSION,
    beta2_divergence_matrix_fourier,
)
from kde_tools.joint_experiments import (
    asymptotic_c_beta,
    asymptotic_grid_size,
    draw_sample,
    exact_expected_q_minimizer,
    inverse_psi_beta,
    local_beta2_log_metric,
    local_custom_beta_metric,
    plot_selected_bandwidth_paths,
    psi_beta,
    richardson_local_metric_from_divergence,
)
from kde_tools.perron import (
    PERRON_SOLVER_VERSION,
    perron_eigenpair_from_divergence,
    perron_peak_index,
    perron_vector_from_divergence,
)
from kde_tools.plotting_functions import plot_bandwidth_distributions
from kde_tools.qhat import expected_gaussian_qhat_log_energy, minimize_qhat
from kde_tools.similarities import beta2_divergence_matrix
from kde_tools.spectral_sweeps import run_l2_point
from threadpoolctl import threadpool_limits

paths = setup_notebook("experiment_1_joint_N_lambda_limit")
code_directory = paths.code
pdf_output_directory = paths.pdfs
ENERGY_SEARCH = EnergySearchConfig()
SEED = 20260823
N_DATA = 200
H_MIN, H_MAX = (0.01, 2.0)
BETA = 2.0
N_VALUES = np.arange(100, 3001, 100)
FREQUENCY_CUTOFF = DEFAULT_FREQUENCY_CUTOFF
FREQUENCY_INTERVALS = DEFAULT_FREQUENCY_INTERVALS
FFT_HALF_WIDTH = np.pi * FREQUENCY_INTERVALS / FREQUENCY_CUTOFF
FFT_SPATIAL_POINTS = 2 * FREQUENCY_INTERVALS
plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=6, suppress=True)


In [ ]:
rng = np.random.default_rng(SEED)
samples = rng.normal(loc=0.0, scale=1.0, size=N_DATA)
endpoint_check = pd.DataFrame(
    {
        "N": N_VALUES,
        "first h": [
            bandwidth_grid_linear_in_psi_beta(H_MIN, H_MAX, BETA, int(N))[0] for N in N_VALUES
        ],
        "last h": [
            bandwidth_grid_linear_in_psi_beta(H_MIN, H_MAX, BETA, int(N))[-1] for N in N_VALUES
        ],
        "Delta log(h)": [np.log(H_MAX / H_MIN) / (N - 1) for N in N_VALUES],
    }
)
sample_summary = pd.DataFrame({"mean": [samples.mean()], "sd": [samples.std(ddof=1)]})
print(f"Generated one sample with n={N_DATA}, using seed={SEED}.")
display(sample_summary.round(6))
display(endpoint_check)


## Continuous large-$N$, large-$\lambda$ prediction

Write $\psi=\log h$. Locally, beta-2 divergence has the quadratic expansion

$$D_2(\widehat f_h,\widehat f_{he^\delta})
=a(h)\delta^2+o(\delta^2),\qquad
a(h)=\frac12\left\|\partial_\psi\widehat f_h\right\|_2^2.$$

The localized continuous kernel has row width proportional to $a(h)^{-1/2}$, so its Perron mass concentrates at

$$h_\infty=\operatorname*{arg\,min}_{h\in[0.01,2]}a(h).$$

For Gaussian KDEs, this coefficient can be evaluated without numerical differentiation. If $d_{rs}=(X_r-X_s)^2$, $q=2h^2$, and $g_q(d)=(2\pi q)^{-1/2}e^{-d/(2q)}$, then

$$a(h)=2h^4\frac1{n^2}\sum_{r,s}g_q(d_{rs})
\left(\frac{3}{4q^2}-\frac{3d_{rs}}{2q^3}
+\frac{d_{rs}^2}{4q^4}\right).$$

In [ ]:
squared_differences = (samples[:, None] - samples[None, :]) ** 2
continuous_minimum = minimize_qhat(samples, H_MIN, H_MAX, r=0.0, config=ENERGY_SEARCH)
if continuous_minimum["diagnostics"]["resolution_stable"] is False:
    raise ArithmeticError("Energy search did not stabilize; increase ENERGY_SEARCH resolution.")
h_infinity = continuous_minimum["global_minimum"]["h"]
metric_grid = np.geomspace(H_MIN, H_MAX, 600)
metric_values = np.array([local_beta2_log_metric(h, squared_differences) for h in metric_grid])
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(metric_grid, metric_values, color="#244c66", linewidth=2.2)
ax.axvline(
    h_infinity,
    color="#c23b22",
    linestyle="--",
    label=f"continuous prediction $h_\\infty={h_infinity:.4f}$",
)
ax.set(
    xscale="log",
    xlabel="bandwidth $h$",
    ylabel="local coefficient $a(h)$",
    title=f"Continuous localized-kernel criterion on $[{H_MIN:g},{H_MAX:g}]$",
)
ax.legend()
plt.show()
print(f"Continuous-limit prediction: h_infinity = {h_infinity:.8f}")


## Numerical method and validation

The project already contains an exact analytic beta-2 calculation. Its cost grows like $N^2n^2$, so the full sweep uses the existing Fourier reconstruction and one matrix product per $N$. The next cell checks that approximation against the exact helper on the smallest grid.

For numerical stability, the Perron vector is computed from the off-diagonal part $A_N=S_N-I_N$. Adding the identity does not change the eigenvectors. Before the centralized `eigh` solve, all off-diagonal entries are multiplied by the same factor so that the largest equals one; this also leaves the eigenvectors unchanged and prevents the $\alpha=3$ regime from becoming numerically indistinguishable from the identity.

In [ ]:
validation_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=H_MIN, h_max=H_MAX, beta=BETA, grid_size=25
)
validation_exact = beta2_divergence_matrix(samples, validation_grid)
validation_fft = beta2_divergence_matrix_fourier(
    samples, validation_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
)
validation_abs_error = np.max(np.abs(validation_fft - validation_exact))
validation_rel_error = validation_abs_error / np.max(validation_exact)
print(f"maximum absolute D2 error: {validation_abs_error:.3e}")
print(f"error relative to max(D2): {validation_rel_error:.3e}")
if validation_rel_error > 2e-05:
    warnings.warn(
        "Increase FREQUENCY_INTERVALS to refine the frequency grid, or FREQUENCY_CUTOFF to reduce truncation."
    )


In [ ]:
LAMBDA_RULES = {
    "$N^{1/2}$ (admissible)": lambda N: N**0.5,
    "$N$ (admissible)": lambda N: float(N),
    "$N^{3/2}$ (admissible)": lambda N: N**1.5,
    "$1$ (not localized)": lambda N: 1.0,
    "$N^2$ (mesh-scale)": lambda N: float(N**2),
    "$N^3$ (collapse)": lambda N: float(N**3),
}
REGIME_ORDER = list(LAMBDA_RULES)
REGIME_COLORS = dict(
    zip(REGIME_ORDER, ["#0072B2", "#009E73", "#CC79A7", "#E69F00", "#7A7A7A", "#D55E00"])
)
rows = []
largest_grid_vectors = {}
start_time = time.perf_counter()
for N in N_VALUES:
    N = int(N)
    h_grid = bandwidth_grid_linear_in_psi_beta(h_min=H_MIN, h_max=H_MAX, beta=BETA, grid_size=N)
    divergence = beta2_divergence_matrix_fourier(
        samples, h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
    )
    reference_index = nearest_bandwidth_grid_index(h_infinity, h_grid)
    neighbor_left = min(reference_index, N - 2)
    neighbor_D = divergence[neighbor_left, neighbor_left + 1]
    endpoint_D = divergence[0, -1]
    for regime, lambda_rule in LAMBDA_RULES.items():
        lambda_value = float(lambda_rule(N))
        vector = perron_vector_from_divergence(divergence, lambda_value)
        selected_index = int(perron_peak_index(vector))
        selected_h = float(h_grid[selected_index])
        rows.append(
            {
                "N": N,
                "regime": regime,
                "lambda": lambda_value,
                "lambda_over_N2": lambda_value / N**2,
                "selected_index": selected_index,
                "selected_h": selected_h,
                "absolute_error": abs(selected_h - h_infinity),
                "global_exponent": lambda_value * endpoint_D,
                "neighbor_exponent": lambda_value * neighbor_D,
                "neighbor_similarity": np.exp(-lambda_value * neighbor_D),
            }
        )
        if N == int(N_VALUES[-1]):
            largest_grid_vectors[regime] = (h_grid.copy(), vector.copy())
results = pd.DataFrame(rows)
results["regime"] = pd.Categorical(results["regime"], categories=REGIME_ORDER, ordered=True)
results = results.sort_values(["regime", "N"]).reset_index(drop=True)
elapsed = time.perf_counter() - start_time
print(f"Completed {len(results)} selector computations in {elapsed:.2f} seconds.")
display(results.pivot(index="N", columns="regime", values="selected_h").round(6))


In [ ]:
plot_ticks = np.array([100, 500, 1000, 1500, 2000, 2500, 3000])


In [ ]:
plot_selected_bandwidth_paths(
    show_continuous_reference=False,
    h_max=H_MAX,
    h_min=H_MIN,
    regime_colors=REGIME_COLORS,
    regime_order=REGIME_ORDER,
    h_infinity=h_infinity,
    metric_grid=metric_grid,
    metric_values=metric_values,
    plot_ticks=plot_ticks,
    results=results,
    squared_differences=squared_differences,
)


In [ ]:
plot_selected_bandwidth_paths(
    show_continuous_reference=True,
    h_max=H_MAX,
    h_min=H_MIN,
    regime_colors=REGIME_COLORS,
    regime_order=REGIME_ORDER,
    h_infinity=h_infinity,
    metric_grid=metric_grid,
    metric_values=metric_values,
    plot_ticks=plot_ticks,
    results=results,
    squared_differences=squared_differences,
)


In [ ]:
CUSTOM_ALPHA_VALUES = np.round(np.arange(1.0, 3.0 + 0.05, 0.5), 1)
CUSTOM_ALPHA_VALUES


## Joint $(N,\lambda_N)$ limit for a user-selected $\beta$

This repeats Figure 3.6 with a freely chosen value of $\beta$. Here $\beta$ determines the coordinate $\psi_\beta$ in which the bandwidth grid is equidistant, and hence the local criterion $\widehat Q_{n,\beta}(h)=h^\beta\lVert\partial_h\widehat f_h\rVert_2^2$. The similarity still uses the same $L^2$ divergence as the original experiment. Change only `CUSTOM_BETA` in the first line of the next cell.

In [ ]:
CUSTOM_BETA = 1 / 7
if not np.isfinite(CUSTOM_BETA) or CUSTOM_BETA <= 0.0:
    raise ValueError("CUSTOM_BETA must be positive and finite.")
CUSTOM_BETA_N_VALUES = N_VALUES.copy()
CUSTOM_ALPHA_VALUES = np.round(np.arange(1.0, 3.0 + 0.05, 0.5), 1)
CUSTOM_BETA_LAMBDA_RULES = {"$N^{0.5}$": lambda N: N**0.5}
for alpha in CUSTOM_ALPHA_VALUES:
    if np.isclose(alpha, 2.0):
        label = "$N^{2.0}$ (mesh-scale)"
    elif alpha > 2.0:
        label = f"$N^{{{alpha:.1f}}}$ (collapse)"
    else:
        label = f"$N^{{{alpha:.1f}}}$"
    CUSTOM_BETA_LAMBDA_RULES[label] = lambda N, alpha=alpha: N**alpha
CUSTOM_BETA_LAMBDA_RULES["$1$ (not localized)"] = lambda N: 1.0
CUSTOM_BETA_REGIME_ORDER = list(CUSTOM_BETA_LAMBDA_RULES)
cmap = plt.colormaps["turbo"]
CUSTOM_BETA_REGIME_COLORS = {
    regime: cmap(i / (len(CUSTOM_BETA_REGIME_ORDER) - 1))
    for i, regime in enumerate(CUSTOM_BETA_REGIME_ORDER)
}
custom_beta_continuous_minimum = minimize_qhat(
    samples, H_MIN, H_MAX, r=1 - CUSTOM_BETA / 2, config=ENERGY_SEARCH
)
if custom_beta_continuous_minimum["diagnostics"]["resolution_stable"] is False:
    raise ArithmeticError("Energy search did not stabilize; increase ENERGY_SEARCH resolution.")
custom_beta_reference_candidates = np.array(
    [H_MIN, custom_beta_continuous_minimum["global_minimum"]["h"], H_MAX]
)
custom_beta_reference_values = np.array(
    [
        local_custom_beta_metric(h, squared_differences, beta=CUSTOM_BETA)
        for h in custom_beta_reference_candidates
    ]
)
custom_beta_h_infinity = float(
    custom_beta_reference_candidates[np.argmin(custom_beta_reference_values)]
)
custom_beta_rows = []
custom_beta_start_time = time.perf_counter()
for current_N in CUSTOM_BETA_N_VALUES:
    current_N = int(current_N)
    current_h_grid = bandwidth_grid_linear_in_psi_beta(
        h_min=H_MIN, h_max=H_MAX, beta=CUSTOM_BETA, grid_size=current_N
    )
    current_divergence = beta2_divergence_matrix_fourier(
        samples, current_h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
    )
    for regime, lambda_rule in CUSTOM_BETA_LAMBDA_RULES.items():
        lambda_value = float(lambda_rule(current_N))
        current_vector = perron_vector_from_divergence(current_divergence, lambda_value)
        selected_index = int(perron_peak_index(current_vector))
        custom_beta_rows.append(
            {
                "beta": CUSTOM_BETA,
                "N": current_N,
                "regime": regime,
                "lambda": lambda_value,
                "selected_index": selected_index,
                "selected_h": float(current_h_grid[selected_index]),
                "continuous_reference": custom_beta_h_infinity,
            }
        )
custom_beta_results = pd.DataFrame(custom_beta_rows)
custom_beta_results["regime"] = pd.Categorical(
    custom_beta_results["regime"], categories=CUSTOM_BETA_REGIME_ORDER, ordered=True
)
custom_beta_results = custom_beta_results.sort_values(["regime", "N"]).reset_index(drop=True)
custom_beta_elapsed = time.perf_counter() - custom_beta_start_time
print(f"Completed beta={CUSTOM_BETA:g} joint-limit sweep in {custom_beta_elapsed:.2f} seconds.")
print(f"Continuous reference for this beta: h_infinity = {custom_beta_h_infinity:.8f}")
display(custom_beta_results.pivot(index="N", columns="regime", values="selected_h").round(6))


In [ ]:
custom_beta_plot_ticks = np.array([100, 500, 1000, 1500, 2000, 2500, 3000])
fig, ax = plt.subplots(figsize=(9.5, 5.8))
for regime in CUSTOM_BETA_REGIME_ORDER:
    subset = custom_beta_results[custom_beta_results["regime"] == regime]
    admissible = "admissible" in regime
    ax.plot(
        subset["N"],
        subset["selected_h"],
        marker="o",
        linewidth=2.2 if admissible else 1.6,
        linestyle="-" if admissible else "--",
        color=CUSTOM_BETA_REGIME_COLORS[regime],
        label=regime,
    )
ax.axhline(
    custom_beta_h_infinity,
    color="black",
    linestyle=":",
    linewidth=2.2,
    label=f"continuous $h_\\infty={custom_beta_h_infinity:.4f}$",
)
ax.set(
    xlabel="number of bandwidths $N$",
    ylabel="selected bandwidth $h_{\\mathrm{spec},N}$",
    title=f"Joint grid-localization limit for $\\beta={CUSTOM_BETA:g}$",
)
ax.set_xticks(custom_beta_plot_ticks, labels=custom_beta_plot_ticks)
ax.legend(ncol=2, frameon=True)
fig.tight_layout()
plt.show()


In [ ]:
MC_BETA = 1 / 7
MC_N = 1000
MC_ALPHA = 1.9
MC_LAMBDA = float(MC_N**MC_ALPHA)
MC_SAMPLE_SIZE = 200
MC_REPETITIONS = 2000
MC_SEED = 12345
mc_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=H_MIN, h_max=H_MAX, beta=MC_BETA, grid_size=MC_N
)
rng = np.random.default_rng(MC_SEED)
mc_rows = []
mc_start_time = time.perf_counter()
for repetition in range(1, MC_REPETITIONS + 1):
    current_samples = draw_sample(rng, MC_SAMPLE_SIZE)
    current_divergence = beta2_divergence_matrix_fourier(
        current_samples, mc_h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
    )
    current_vector = perron_vector_from_divergence(current_divergence, MC_LAMBDA)
    selected_index = int(perron_peak_index(current_vector))
    selected_h = float(mc_h_grid[selected_index])
    mc_rows.append(
        {
            "repetition": repetition,
            "n": MC_SAMPLE_SIZE,
            "N": MC_N,
            "beta": MC_BETA,
            "alpha": MC_ALPHA,
            "lambda": MC_LAMBDA,
            "selected_index": selected_index,
            "selected_h": selected_h,
        }
    )
    if repetition % 10 == 0:
        elapsed = time.perf_counter() - mc_start_time
        print(
            f"Completed {repetition:3d}/{MC_REPETITIONS} repetitions after {elapsed:.1f} seconds."
        )
mc_bandwidth_results = pd.DataFrame(mc_rows)
mc_elapsed = time.perf_counter() - mc_start_time
print()
print(f"Completed all {MC_REPETITIONS} repetitions in {mc_elapsed:.2f} seconds.")
print(f"beta     = {MC_BETA:.8f}")
print(f"N        = {MC_N}")
print(f"alpha    = {MC_ALPHA}")
print(f"lambda_N = {MC_LAMBDA:.6e}")
selected_bandwidths = mc_bandwidth_results["selected_h"].to_numpy()
mc_mean = float(np.mean(selected_bandwidths))
mc_median = float(np.median(selected_bandwidths))
mc_std = float(np.std(selected_bandwidths, ddof=1))
mc_q025, mc_q25, mc_q75, mc_q975 = np.quantile(selected_bandwidths, [0.025, 0.25, 0.75, 0.975])
print()
print("Selected bandwidth distribution")
print("--------------------------------")
print(f"mean       = {mc_mean:.8f}")
print(f"median     = {mc_median:.8f}")
print(f"std        = {mc_std:.8f}")
print(f"2.5%       = {mc_q025:.8f}")
print(f"25%        = {mc_q25:.8f}")
print(f"75%        = {mc_q75:.8f}")
print(f"97.5%      = {mc_q975:.8f}")
display(mc_bandwidth_results)
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(selected_bandwidths, bins=20, density=True, alpha=0.75, edgecolor="black")
ax.axvline(mc_mean, linestyle="--", linewidth=1.8, label=f"mean $={mc_mean:.4f}$")
ax.axvline(mc_median, linestyle=":", linewidth=1.8, label=f"median $={mc_median:.4f}$")
ax.set_xlabel("selected bandwidth $\\hat h_{\\rm spec}$")
ax.set_ylabel("density")
ax.set_title(
    f"Monte Carlo distribution of $\\hat h_{{\\rm spec}}$\n$n={MC_SAMPLE_SIZE}$, $N={MC_N}$, $\\beta=1/7$, $\\lambda_N=N^{{{MC_ALPHA}}}$"
)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
sorted_h = np.sort(selected_bandwidths)
empirical_cdf = np.arange(1, MC_REPETITIONS + 1) / MC_REPETITIONS
fig, ax = plt.subplots(figsize=(10, 6))
ax.step(sorted_h, empirical_cdf, where="post", linewidth=1.8)
ax.set_xlabel("selected bandwidth $\\hat h_{\\rm spec}$")
ax.set_ylabel("empirical CDF")
ax.set_title(f"Empirical distribution of $\\hat h_{{\\rm spec}}$, $n={MC_SAMPLE_SIZE}$")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
selected_indices = mc_bandwidth_results["selected_index"].to_numpy(dtype=int)
selected_counts = np.bincount(selected_indices, minlength=len(mc_h_grid)).astype(float)
mean_index = int(np.argmin(np.abs(mc_h_grid - mc_mean)))
median_index = int(np.argmin(np.abs(mc_h_grid - mc_median)))
reference_lines = [
    (f"mean $\\hat h_{{\\rm spec}}={mc_mean:.4f}$", mean_index, "#D55E00"),
    (f"median $\\hat h_{{\\rm spec}}={mc_median:.4f}$", median_index, "#009E73"),
]
fig = plot_bandwidth_distributions(
    h_grid=mc_h_grid,
    distributions={f"$\\beta=1/7,\\ N={MC_N},\\ \\lambda_N=N^{{{MC_ALPHA}}}$": selected_counts},
    reference_lines=reference_lines,
    tick_count=9,
    width=1100,
    height=600,
)
fig.update_layout(
    title=f"Distribution of selected bandwidths over {MC_REPETITIONS} samples<br><sup>$n={MC_SAMPLE_SIZE}$, $N={MC_N}$, $\\beta=1/7$, $\\lambda_N=N^{{{MC_ALPHA}}}$</sup>"
)
fig.show()


In [ ]:
MESH_STRESS_N = 400
MESH_TARGET_H = 0.7
MESH_ALPHA_VALUES = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0])
MESH_ALPHA_COLORS = plt.colormaps["tab20"](np.arange(len(MESH_ALPHA_VALUES)))
MESH_GAP_FRACTIONS = np.array([1.0, 0.9, 0.75, 0.6, 0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15, 0.1])
mesh_base_grid = bandwidth_grid_linear_in_psi_beta(H_MIN, H_MAX, beta=BETA, grid_size=MESH_STRESS_N)
mesh_base_psi = np.log(mesh_base_grid)
mesh_delta_psi = mesh_base_psi[1] - mesh_base_psi[0]
mesh_close_pair_left = int(np.argmin(np.abs(mesh_base_grid - MESH_TARGET_H)))
mesh_close_pair_left = min(mesh_close_pair_left, MESH_STRESS_N - 2)
mesh_rows = []
mesh_alpha3_profiles = {}
for gap_fraction in MESH_GAP_FRACTIONS:
    perturbed_psi = mesh_base_psi.copy()
    perturbed_psi[mesh_close_pair_left + 1] = (
        perturbed_psi[mesh_close_pair_left] + gap_fraction * mesh_delta_psi
    )
    h_grid = np.exp(perturbed_psi)
    divergence = beta2_divergence_matrix_fourier(
        samples, h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
    )
    for alpha in MESH_ALPHA_VALUES:
        alpha = float(alpha)
        lambda_value = float(MESH_STRESS_N**alpha)
        vector = perron_vector_from_divergence(divergence, lambda_value)
        selected_index = int(perron_peak_index(vector))
        mesh_rows.append(
            {
                "gap_fraction": gap_fraction,
                "alpha": alpha,
                "lambda": lambda_value,
                "selected_h": float(h_grid[selected_index]),
                "selected_index": selected_index,
                "close_pair_h_left": float(h_grid[mesh_close_pair_left]),
                "close_pair_h_right": float(h_grid[mesh_close_pair_left + 1]),
                "close_pair_similarity": float(
                    np.exp(
                        -lambda_value * divergence[mesh_close_pair_left, mesh_close_pair_left + 1]
                    )
                ),
            }
        )
        if np.isclose(alpha, 3.0) and gap_fraction in {1.0, 0.2}:
            mesh_alpha3_profiles[gap_fraction] = (h_grid.copy(), vector.copy())
mesh_stress_results = pd.DataFrame(mesh_rows)
mesh_selected_table = mesh_stress_results.pivot(
    index="gap_fraction", columns="alpha", values="selected_h"
)
print(f"Completed {len(mesh_stress_results)} mesh-stress selector computations.")
display(mesh_selected_table.loc[[1.0, 0.6, 0.4, 0.3, 0.2, 0.1]].round(6))


In [ ]:
CUSTOM_BETA = 1 / 7
if not np.isfinite(CUSTOM_BETA) or CUSTOM_BETA <= 0.0:
    raise ValueError("CUSTOM_BETA must be positive and finite.")
CUSTOM_BETA_N_VALUES = N_VALUES.copy()
CUSTOM_ALPHA_VALUES = np.round(np.arange(1.0, 3.01, 0.5), 1)
CUSTOM_BETA_LAMBDA_RULES = {"$N^{0.5}$": lambda N: N**0.5}
for alpha in CUSTOM_ALPHA_VALUES:
    if np.isclose(alpha, 2.0):
        label = "$N^{2.0}$ (mesh-scale)"
    elif alpha > 2.0:
        label = f"$N^{{{alpha:.1f}}}$ (collapse)"
    else:
        label = f"$N^{{{alpha:.1f}}}$"
    CUSTOM_BETA_LAMBDA_RULES[label] = lambda N, alpha=alpha: N**alpha
CUSTOM_BETA_LAMBDA_RULES["$1$ (not localized)"] = lambda N: 1.0
CUSTOM_BETA_REGIME_ORDER = list(CUSTOM_BETA_LAMBDA_RULES)
cmap = plt.colormaps["turbo"]
CUSTOM_BETA_REGIME_COLORS = {
    regime: cmap(i / (len(CUSTOM_BETA_REGIME_ORDER) - 1))
    for i, regime in enumerate(CUSTOM_BETA_REGIME_ORDER)
}
custom_beta_continuous_minimum = minimize_qhat(
    samples, H_MIN, H_MAX, r=1 - CUSTOM_BETA / 2, config=ENERGY_SEARCH
)
if custom_beta_continuous_minimum["diagnostics"]["resolution_stable"] is False:
    raise ArithmeticError("Energy search did not stabilize; increase ENERGY_SEARCH resolution.")
custom_beta_reference_candidates = np.array(
    [H_MIN, custom_beta_continuous_minimum["global_minimum"]["h"], H_MAX]
)
custom_beta_reference_values = np.array(
    [
        local_custom_beta_metric(h, squared_differences, beta=CUSTOM_BETA)
        for h in custom_beta_reference_candidates
    ]
)
custom_beta_h_infinity = float(
    custom_beta_reference_candidates[np.argmin(custom_beta_reference_values)]
)
diagnostic_rows = []
diagnostic_start_time = time.perf_counter()
psi_min = psi_beta(H_MIN, CUSTOM_BETA)
psi_max = psi_beta(H_MAX, CUSTOM_BETA)
for current_N in CUSTOM_BETA_N_VALUES:
    current_N = int(current_N)
    current_h_grid = bandwidth_grid_linear_in_psi_beta(
        h_min=H_MIN, h_max=H_MAX, beta=CUSTOM_BETA, grid_size=current_N
    )
    delta_psi = (psi_max - psi_min) / (current_N - 1)
    current_divergence = beta2_divergence_matrix_fourier(
        samples, current_h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
    )
    reference_index = int(np.argmin(np.abs(current_h_grid - custom_beta_h_infinity)))
    reference_h = float(current_h_grid[reference_index])
    Q_i = local_custom_beta_metric(reference_h, squared_differences, beta=CUSTOM_BETA)
    if not np.isfinite(Q_i) or Q_i <= 0.0:
        raise RuntimeError(f"Invalid local metric Q_i={Q_i} at N={current_N}.")
    grid_offsets = (np.arange(current_N) - reference_index) * delta_psi
    divergence_row = np.maximum(current_divergence[reference_index, :], 0.0)
    for regime, lambda_rule in CUSTOM_BETA_LAMBDA_RULES.items():
        lambda_value = float(lambda_rule(current_N))
        similarity_row = np.exp(-lambda_value * divergence_row)
        actual_row_sum = float(np.sum(similarity_row))
        diagonal_mass = float(similarity_row[reference_index])
        off_diagonal_mass = float(actual_row_sum - diagonal_mass)
        gaussian_row = np.exp(-lambda_value * Q_i * grid_offsets**2)
        gaussian_row_sum = float(np.sum(gaussian_row))
        continuous_integral = float(np.sqrt(np.pi / (lambda_value * Q_i)))
        actual_integral_ratio = float(delta_psi * actual_row_sum / continuous_integral)
        gaussian_quadrature_ratio = float(delta_psi * gaussian_row_sum / continuous_integral)
        local_approximation_ratio = float(actual_row_sum / gaussian_row_sum)
        mesh_to_width = float(np.sqrt(lambda_value * Q_i) * delta_psi)
        effective_grid_points = float(1.0 / mesh_to_width)
        nearest_values = []
        if reference_index > 0:
            nearest_values.append(similarity_row[reference_index - 1])
        if reference_index < current_N - 1:
            nearest_values.append(similarity_row[reference_index + 1])
        nearest_neighbor_similarity = float(np.mean(nearest_values))
        diagnostic_rows.append(
            {
                "beta": CUSTOM_BETA,
                "N": current_N,
                "regime": regime,
                "lambda": lambda_value,
                "reference_index": reference_index,
                "reference_h": reference_h,
                "Q_i": Q_i,
                "delta_psi": delta_psi,
                "mesh_to_width": mesh_to_width,
                "effective_grid_points": effective_grid_points,
                "actual_row_sum": actual_row_sum,
                "gaussian_row_sum": gaussian_row_sum,
                "continuous_integral": continuous_integral,
                "actual_integral_ratio": actual_integral_ratio,
                "gaussian_quadrature_ratio": gaussian_quadrature_ratio,
                "local_approximation_ratio": local_approximation_ratio,
                "off_diagonal_mass": off_diagonal_mass,
                "nearest_neighbor_similarity": nearest_neighbor_similarity,
            }
        )
diagnostic_results = pd.DataFrame(diagnostic_rows)
diagnostic_results["regime"] = pd.Categorical(
    diagnostic_results["regime"], categories=CUSTOM_BETA_REGIME_ORDER, ordered=True
)
diagnostic_results = diagnostic_results.sort_values(["regime", "N"]).reset_index(drop=True)
diagnostic_elapsed = time.perf_counter() - diagnostic_start_time
print(f"Completed diagnostic sweep for beta={CUSTOM_BETA:g} in {diagnostic_elapsed:.2f} seconds.")
print(f"Continuous reference: h_infinity = {custom_beta_h_infinity:.8f}")
display(
    diagnostic_results.pivot(index="N", columns="regime", values="gaussian_quadrature_ratio").round(
        6
    )
)
fig, ax = plt.subplots(figsize=(10, 6))
for regime in CUSTOM_BETA_REGIME_ORDER:
    subset = diagnostic_results[diagnostic_results["regime"] == regime]
    ax.plot(
        subset["N"],
        subset["gaussian_quadrature_ratio"],
        marker="o",
        linewidth=1.8,
        markersize=4,
        label=regime,
        color=CUSTOM_BETA_REGIME_COLORS[regime],
    )
ax.axhline(1.0, linestyle="--", linewidth=1.5, color="black", label="continuum integral")
ax.set_xscale("log")
ax.set_xlabel("number of bandwidths $N$")
ax.set_ylabel("$\\Delta_\\psi \\sum_j g_i(\\delta_{ij})\\,/\\,\\sqrt{\\pi/(\\lambda Q_i)}$")
ax.set_title(f"Failure of the continuum quadrature regime, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(10, 6))
for regime in CUSTOM_BETA_REGIME_ORDER:
    subset = diagnostic_results[diagnostic_results["regime"] == regime]
    ax.plot(
        subset["N"],
        subset["mesh_to_width"],
        marker="o",
        linewidth=1.8,
        markersize=4,
        label=regime,
        color=CUSTOM_BETA_REGIME_COLORS[regime],
    )
ax.axhline(1.0, linestyle="--", linewidth=1.5, color="black")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("number of bandwidths $N$")
ax.set_ylabel("$\\sqrt{\\lambda Q_i}\\,\\Delta_\\psi$")
ax.set_title(f"Mesh scale versus localization scale, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(10, 6))
for regime in CUSTOM_BETA_REGIME_ORDER:
    subset = diagnostic_results[diagnostic_results["regime"] == regime]
    values = np.maximum(subset["off_diagonal_mass"].to_numpy(), 1e-300)
    ax.plot(
        subset["N"],
        values,
        marker="o",
        linewidth=1.8,
        markersize=4,
        label=regime,
        color=CUSTOM_BETA_REGIME_COLORS[regime],
    )
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("number of bandwidths $N$")
ax.set_ylabel("$\\sum_{j\\neq i} S_{ij}$")
ax.set_title(f"Off-diagonal similarity mass, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(10, 6))
for regime in CUSTOM_BETA_REGIME_ORDER:
    subset = diagnostic_results[diagnostic_results["regime"] == regime]
    values = np.maximum(subset["nearest_neighbor_similarity"].to_numpy(), 1e-300)
    ax.plot(
        subset["N"],
        values,
        marker="o",
        linewidth=1.8,
        markersize=4,
        label=regime,
        color=CUSTOM_BETA_REGIME_COLORS[regime],
    )
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("number of bandwidths $N$")
ax.set_ylabel("mean nearest-neighbor similarity")
ax.set_title(f"Nearest-neighbor collapse, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
criterion_results = diagnostic_results.copy()
criterion_results["alpha"] = np.log(criterion_results["lambda"]) / np.log(criterion_results["N"])
criterion_results["alpha"] = criterion_results["alpha"].round(10)
criterion_results = criterion_results[
    (criterion_results["alpha"] >= 1.0) & (criterion_results["alpha"] <= 3.0)
].copy()
eps = 1e-300
criterion_results["local_ratio_safe"] = np.maximum(
    criterion_results["local_approximation_ratio"], eps
)
criterion_results["mesh_ratio_safe"] = np.maximum(
    criterion_results["gaussian_quadrature_ratio"], eps
)
criterion_results["E_local"] = np.abs(np.log(criterion_results["local_ratio_safe"]))
criterion_results["E_mesh"] = np.abs(np.log(criterion_results["mesh_ratio_safe"]))
criterion_results["C"] = np.sqrt(
    criterion_results["E_local"] ** 2 + criterion_results["E_mesh"] ** 2
)
fig, ax = plt.subplots(figsize=(11, 7))
for current_N in sorted(criterion_results["N"].unique()):
    subset = criterion_results[criterion_results["N"] == current_N].sort_values("alpha")
    ax.plot(
        subset["alpha"],
        subset["C"],
        marker="o",
        markersize=4,
        linewidth=1.5,
        label=f"$N={current_N}$",
    )
ax.axvline(2.0, linestyle="--", linewidth=1.5, color="black", label="critical $\\alpha=2$")
ax.set_xlabel("exponent $\\alpha$ in $\\lambda_N=N^\\alpha$")
ax.set_ylabel("$C_N(\\alpha)$")
ax.set_title(f"Localization--mesh criterion, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=8, ncol=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
largest_N = int(criterion_results["N"].max())
largest_N_results = (
    criterion_results[criterion_results["N"] == largest_N].sort_values("alpha").copy()
)
optimal_row = largest_N_results.loc[largest_N_results["C"].idxmin()]
optimal_alpha = float(optimal_row["alpha"])
optimal_C = float(optimal_row["C"])
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(largest_N_results["alpha"], largest_N_results["C"], marker="o", markersize=5, linewidth=2)
ax.axvline(2.0, linestyle="--", linewidth=1.5, color="black", label="critical $\\alpha=2$")
ax.axvline(
    optimal_alpha,
    linestyle=":",
    linewidth=1.8,
    color="black",
    label=f"$\\alpha_{{N}}^\\star={optimal_alpha:.1f}$",
)
ax.scatter([optimal_alpha], [optimal_C], s=70, zorder=5)
ax.set_xlabel("exponent $\\alpha$ in $\\lambda_N=N^\\alpha$")
ax.set_ylabel(f"$C_{{{largest_N}}}(\\alpha)$")
ax.set_title(f"Criterion at $N={largest_N}$, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
print(
    f"For N={largest_N}, the minimizing exponent is alpha* = {optimal_alpha:.1f}, with C = {optimal_C:.8g}."
)
optimal_alpha_by_N = (
    criterion_results.loc[
        criterion_results.groupby("N")["C"].idxmin(), ["N", "alpha", "C", "E_local", "E_mesh"]
    ]
    .sort_values("N")
    .reset_index(drop=True)
)
optimal_alpha_by_N = optimal_alpha_by_N.rename(columns={"alpha": "optimal_alpha", "C": "minimum_C"})
display(optimal_alpha_by_N.round(8))


### Zoom on the positive-power regimes

Here we retain all positive-power exponents $\alpha\in\{1/2,1,3/2,2,3\}$, restrict attention to $N\geq1000$, and zoom in around the continuous reference $h_\infty$. Solid lines denote the three admissible regimes $\alpha<2$; dashed lines denote the mesh-scale case $\alpha=2$ and the collapsing case $\alpha=3$. Their visual proximity to $h_\infty$ should therefore not be interpreted as convergence to the same continuous localized-kernel limit. The small oscillations are caused by projecting the target onto a finite bandwidth grid.


In [ ]:
FOCUSED_REGIMES = [
    REGIME_ORDER[0],
    REGIME_ORDER[1],
    REGIME_ORDER[2],
    REGIME_ORDER[4],
    REGIME_ORDER[5],
]
ZOOM_N_MIN = 1000
zoom_results = results[
    results["regime"].isin(FOCUSED_REGIMES) & (results["N"] >= ZOOM_N_MIN)
].copy()
REGIME_MARKERS = dict(zip(FOCUSED_REGIMES, ["o", "s", "^", "D", "X"]))
fig, ax = plt.subplots(figsize=(9.5, 5.8))
for regime in FOCUSED_REGIMES:
    subset = zoom_results[zoom_results["regime"] == regime]
    admissible = "admissible" in regime
    ax.plot(
        subset["N"],
        subset["selected_h"],
        marker=REGIME_MARKERS[regime],
        markersize=5.5,
        linewidth=2.2 if admissible else 1.7,
        linestyle="-" if admissible else "--",
        markerfacecolor=REGIME_COLORS[regime] if admissible else "white",
        markeredgewidth=1.4,
        color=REGIME_COLORS[regime],
        label=regime,
    )
zoom_min = min(h_infinity, zoom_results["selected_h"].min())
zoom_max = max(h_infinity, zoom_results["selected_h"].max())
zoom_padding = 0.06 * (zoom_max - zoom_min)
ax.axhline(
    h_infinity,
    color="black",
    linestyle=":",
    linewidth=2.0,
    label=f"continuous $h_\\infty={h_infinity:.4f}$",
)
ax.set_ylim(zoom_min - zoom_padding, zoom_max + zoom_padding)
ax.set(
    xlabel="number of bandwidths $N$",
    ylabel="selected bandwidth $h_{\\mathrm{spec},N}$",
    title="Positive-power regimes near the continuous reference",
)
ax.set_xticks(plot_ticks[plot_ticks >= ZOOM_N_MIN], labels=plot_ticks[plot_ticks >= ZOOM_N_MIN])
ax.legend(ncol=2, frameon=True)
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.8))
for regime in REGIME_ORDER:
    subset = results[results["regime"] == regime]
    ax.plot(
        subset["N"],
        subset["neighbor_exponent"],
        marker="o",
        linewidth=2.0,
        color=REGIME_COLORS[regime],
        label=regime,
    )
ax.axhline(1.0, color="black", linestyle=":", linewidth=1.5)
ax.set(
    yscale="log",
    xlabel="number of bandwidths $N$",
    ylabel="$\\lambda_ND_{i,i+1}$ near $h_\\infty$",
    title="Mesh-resolution diagnostic (must tend to 0)",
)
ax.set_xticks(plot_ticks, labels=plot_ticks)
ax.legend(ncol=2, frameon=True)
fig.tight_layout()
plt.show()


In [ ]:
final_N = int(N_VALUES[-1])
final_diagnostics = results[results["N"] == final_N][
    [
        "regime",
        "lambda",
        "lambda_over_N2",
        "selected_h",
        "absolute_error",
        "global_exponent",
        "neighbor_exponent",
        "neighbor_similarity",
    ]
].copy()
admissible_final = final_diagnostics[final_diagnostics["regime"].isin(REGIME_ORDER[:3])]
admissible_spread = admissible_final["selected_h"].max() - admissible_final["selected_h"].min()
display(final_diagnostics.round(7))
print(f"At N={final_N}, the admissible selectors span {admissible_spread:.6f} in bandwidth.")
fig, ax = plt.subplots(figsize=(10.5, 5.2))
for regime in REGIME_ORDER:
    h_grid, vector = largest_grid_vectors[regime]
    admissible = "admissible" in regime
    ax.plot(
        h_grid,
        vector / vector.max(),
        color=REGIME_COLORS[regime],
        linewidth=2.0 if admissible else 1.5,
        linestyle="-" if admissible else "--",
        label=regime,
    )
ax.axvline(h_infinity, color="black", linestyle=":", linewidth=2.0)
ax.set(
    xscale="log",
    xlabel="bandwidth $h$",
    ylabel="Perron component / maximum",
    title=f"Perron profiles at the finest grid ($N={final_N}$)",
)
ax.legend(ncol=2)
plt.show()


## Mesh-robustness stress test for the six regimes

We use the same six exponents as above:

$$\alpha\in\{0,1/2,1,3/2,2,3\},\qquad \lambda_N=N^\alpha.$$

For each exponent, one point in an otherwise uniform $\psi_2=\log h$ grid is moved toward its left neighbor near $h=0.7$, far from the continuous prediction $h_\infty$. The endpoints, number of candidates, sample, bandwidth family, and all other grid points remain fixed.

The retained-gap fraction runs from one (the original uniform grid) down to $0.1$. A jump of the selected bandwidth toward $0.7$ indicates that the Perron vector has locked onto the artificial close pair.

**Scope.** This is a robustness stress test, not a counterexample under the theorem's exact uniform-grid assumptions.


In [ ]:
MESH_STRESS_N = 400
MESH_TARGET_H = 0.7
MESH_ALPHA_VALUES = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0])
MESH_ALPHA_COLORS = plt.colormaps["tab20"](np.arange(len(MESH_ALPHA_VALUES)))
MESH_GAP_FRACTIONS = np.array([1.0, 0.9, 0.75, 0.6, 0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15, 0.1])
mesh_base_grid = bandwidth_grid_linear_in_psi_beta(H_MIN, H_MAX, beta=BETA, grid_size=MESH_STRESS_N)
mesh_base_psi = np.log(mesh_base_grid)
mesh_delta_psi = mesh_base_psi[1] - mesh_base_psi[0]
mesh_close_pair_left = int(np.argmin(np.abs(mesh_base_grid - MESH_TARGET_H)))
mesh_close_pair_left = min(mesh_close_pair_left, MESH_STRESS_N - 2)
mesh_rows = []
mesh_alpha3_profiles = {}
for gap_fraction in MESH_GAP_FRACTIONS:
    perturbed_psi = mesh_base_psi.copy()
    perturbed_psi[mesh_close_pair_left + 1] = (
        perturbed_psi[mesh_close_pair_left] + gap_fraction * mesh_delta_psi
    )
    h_grid = np.exp(perturbed_psi)
    divergence = beta2_divergence_matrix_fourier(
        samples, h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
    )
    for alpha in MESH_ALPHA_VALUES:
        alpha = float(alpha)
        lambda_value = float(MESH_STRESS_N**alpha)
        vector = perron_vector_from_divergence(divergence, lambda_value)
        selected_index = int(perron_peak_index(vector))
        mesh_rows.append(
            {
                "gap_fraction": gap_fraction,
                "alpha": alpha,
                "lambda": lambda_value,
                "selected_h": float(h_grid[selected_index]),
                "selected_index": selected_index,
                "close_pair_h_left": float(h_grid[mesh_close_pair_left]),
                "close_pair_h_right": float(h_grid[mesh_close_pair_left + 1]),
                "close_pair_similarity": float(
                    np.exp(
                        -lambda_value * divergence[mesh_close_pair_left, mesh_close_pair_left + 1]
                    )
                ),
            }
        )
        if np.isclose(alpha, 3.0) and gap_fraction in {1.0, 0.2}:
            mesh_alpha3_profiles[gap_fraction] = (h_grid.copy(), vector.copy())
mesh_stress_results = pd.DataFrame(mesh_rows)
mesh_selected_table = mesh_stress_results.pivot(
    index="gap_fraction", columns="alpha", values="selected_h"
)
print(f"Completed {len(mesh_stress_results)} mesh-stress selector computations.")
display(mesh_selected_table.loc[[1.0, 0.6, 0.4, 0.3, 0.2, 0.1]].round(6))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.4))
for alpha_index, alpha in enumerate(MESH_ALPHA_VALUES):
    subset = mesh_stress_results[mesh_stress_results["alpha"] == alpha].sort_values(
        "gap_fraction", ascending=False
    )
    axes[0].plot(
        subset["gap_fraction"],
        subset["selected_h"],
        color=MESH_ALPHA_COLORS[alpha_index],
        linewidth=1.8,
        marker="o",
        markersize=4.8,
        markeredgecolor="white",
        markeredgewidth=0.35,
        label=f"$\\alpha={alpha:g}$",
    )
axes[0].axhline(
    h_infinity, color="black", linestyle=":", linewidth=2.0, label=f"$h_\\infty={h_infinity:.4f}$"
)
axes[0].axhline(
    MESH_TARGET_H,
    color="#D55E00",
    linestyle=":",
    linewidth=1.5,
    label=f"perturbed region $h\\approx{MESH_TARGET_H}$",
)
axes[0].invert_xaxis()
axes[0].set(
    xlabel="retained fraction of the standard local grid gap",
    ylabel="selected bandwidth $h_{\\mathrm{spec}}$",
    title="Mesh sensitivity for $\\alpha\\in\\{0,1/2,1,3/2,2,3\\}$",
)
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=4, fontsize=8.5, frameon=True)
alpha3_color = MESH_ALPHA_COLORS[-1]
for gap_fraction, linestyle in [(1.0, "-"), (0.2, "--")]:
    h_grid, vector = mesh_alpha3_profiles[gap_fraction]
    axes[1].plot(
        h_grid,
        vector / vector.max(),
        color=alpha3_color,
        linestyle=linestyle,
        linewidth=2.6,
        label=(
            "$\\alpha=3$, uniform grid" if gap_fraction == 1.0 else "$\\alpha=3$, local gap = 20%"
        ),
    )
axes[1].axvline(
    h_infinity, color="black", linestyle=":", linewidth=2.0, label=f"$h_\\infty={h_infinity:.4f}$"
)
axes[1].axvline(
    MESH_TARGET_H,
    color="#D55E00",
    linestyle=":",
    linewidth=1.5,
    label=f"perturbed region $h\\approx{MESH_TARGET_H}$",
)
axes[1].set(
    xscale="log",
    xlabel="bandwidth $h$",
    ylabel="Perron component / maximum",
    title="$\\alpha=3$ locks onto the artificial close pair",
)
axes[1].legend()
fig.suptitle(
    f"Mesh-robustness stress test on one fixed sample ($N={MESH_STRESS_N}$, $n={N_DATA}$, one perturbed grid point)",
    fontsize=14,
)
fig.subplots_adjust(bottom=0.31, wspace=0.22)
plt.show()


In [ ]:
MESH_TARGET_H = 0.7
MESH_STRESS_N_VALUES = np.array([200, 400, 800, 1200])
MESH_ALPHA_VALUES = np.concatenate([np.array([0.0, 0.5]), np.round(np.arange(1.0, 3.01, 0.1), 1)])
MESH_GAP_FRACTIONS = np.array([1.0, 0.9, 0.75, 0.6, 0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15, 0.1])
mesh_cmap = plt.colormaps["turbo"]
MESH_ALPHA_COLORS = {
    float(alpha): mesh_cmap(i / (len(MESH_ALPHA_VALUES) - 1))
    for i, alpha in enumerate(MESH_ALPHA_VALUES)
}
mesh_rows = []
mesh_start_time = time.perf_counter()
for mesh_N in MESH_STRESS_N_VALUES:
    mesh_N = int(mesh_N)
    mesh_base_grid = bandwidth_grid_linear_in_psi_beta(
        H_MIN, H_MAX, beta=CUSTOM_BETA, grid_size=mesh_N
    )
    mesh_base_psi = np.array([psi_beta(h, CUSTOM_BETA) for h in mesh_base_grid])
    mesh_delta_psi = float(mesh_base_psi[1] - mesh_base_psi[0])
    mesh_close_pair_left = int(np.argmin(np.abs(mesh_base_grid - MESH_TARGET_H)))
    mesh_close_pair_left = max(1, mesh_close_pair_left)
    mesh_close_pair_left = min(mesh_close_pair_left, mesh_N - 3)
    for gap_fraction in MESH_GAP_FRACTIONS:
        gap_fraction = float(gap_fraction)
        perturbed_psi = mesh_base_psi.copy()
        perturbed_psi[mesh_close_pair_left + 1] = (
            perturbed_psi[mesh_close_pair_left] + gap_fraction * mesh_delta_psi
        )
        h_grid = np.array([inverse_psi_beta(psi, CUSTOM_BETA) for psi in perturbed_psi])
        divergence = beta2_divergence_matrix_fourier(
            samples, h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
        )
        i = mesh_close_pair_left
        close_pair_divergence = float(max(divergence[i, i + 1], 0.0))
        regular_pair_divergence = float(max(divergence[i - 1, i], 0.0))
        Q_i = local_custom_beta_metric(float(h_grid[i]), squared_differences, beta=CUSTOM_BETA)
        for alpha in MESH_ALPHA_VALUES:
            alpha = float(alpha)
            lambda_value = float(mesh_N**alpha)
            vector = perron_vector_from_divergence(divergence, lambda_value)
            selected_index = int(perron_peak_index(vector))
            selected_h = float(h_grid[selected_index])
            close_pair_similarity = float(np.exp(-lambda_value * close_pair_divergence))
            regular_pair_similarity = float(np.exp(-lambda_value * regular_pair_divergence))
            log_similarity_ratio = float(
                lambda_value * (regular_pair_divergence - close_pair_divergence)
            )
            predicted_log_similarity_ratio = float(
                lambda_value * Q_i * mesh_delta_psi**2 * (1.0 - gap_fraction**2)
            )
            lambda_delta_psi_squared = float(lambda_value * mesh_delta_psi**2)
            mesh_rows.append(
                {
                    "N": mesh_N,
                    "gap_fraction": gap_fraction,
                    "alpha": alpha,
                    "lambda": lambda_value,
                    "selected_h": selected_h,
                    "selected_index": selected_index,
                    "close_pair_h_left": float(h_grid[i]),
                    "close_pair_h_right": float(h_grid[i + 1]),
                    "delta_psi": mesh_delta_psi,
                    "Q_i": Q_i,
                    "close_pair_divergence": close_pair_divergence,
                    "regular_pair_divergence": regular_pair_divergence,
                    "close_pair_similarity": close_pair_similarity,
                    "regular_pair_similarity": regular_pair_similarity,
                    "log_similarity_ratio": log_similarity_ratio,
                    "predicted_log_similarity_ratio": predicted_log_similarity_ratio,
                    "lambda_delta_psi_squared": lambda_delta_psi_squared,
                }
            )
mesh_stress_results = pd.DataFrame(mesh_rows)
mesh_elapsed = time.perf_counter() - mesh_start_time
print(
    f"Completed {len(mesh_stress_results)} mesh-stress selector computations in {mesh_elapsed:.2f} seconds."
)
mesh_baseline = mesh_stress_results[np.isclose(mesh_stress_results["gap_fraction"], 1.0)][
    ["N", "alpha", "selected_h"]
].rename(columns={"selected_h": "baseline_selected_h"})
mesh_stress_results = mesh_stress_results.merge(mesh_baseline, on=["N", "alpha"], how="left")
mesh_stress_results["selector_sensitivity"] = np.abs(
    mesh_stress_results["selected_h"] - mesh_stress_results["baseline_selected_h"]
)
mesh_max_sensitivity = (
    mesh_stress_results.groupby(["N", "alpha"], as_index=False)["selector_sensitivity"]
    .max()
    .rename(columns={"selector_sensitivity": "max_selector_sensitivity"})
)
display(
    mesh_max_sensitivity.pivot(index="alpha", columns="N", values="max_selector_sensitivity").round(
        6
    )
)
DISPLAY_N = 400
display_data = mesh_stress_results[mesh_stress_results["N"] == DISPLAY_N]
fig, ax = plt.subplots(figsize=(13, 8))
for alpha in MESH_ALPHA_VALUES:
    subset = display_data[np.isclose(display_data["alpha"], alpha)].sort_values("gap_fraction")
    ax.plot(
        subset["gap_fraction"],
        subset["selector_sensitivity"],
        marker="o",
        markersize=4,
        linewidth=1.4,
        label=f"$\\alpha={alpha:.1f}$",
        color=MESH_ALPHA_COLORS[float(alpha)],
    )
ax.set_xlabel("compressed gap fraction $c$")
ax.set_ylabel("$|h_{\\rm spec}(c)-h_{\\rm spec}(1)|$")
ax.set_title(f"Mesh sensitivity at $N={DISPLAY_N}$, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=8, ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(11, 7))
for current_N in sorted(mesh_max_sensitivity["N"].unique()):
    subset = mesh_max_sensitivity[mesh_max_sensitivity["N"] == current_N].sort_values("alpha")
    ax.plot(
        subset["alpha"],
        subset["max_selector_sensitivity"],
        marker="o",
        markersize=4,
        linewidth=1.8,
        label=f"$N={current_N}$",
    )
ax.axvline(2.0, linestyle="--", linewidth=1.5, color="black", label="critical $\\alpha=2$")
ax.set_xlabel("exponent $\\alpha$ in $\\lambda_N=N^\\alpha$")
ax.set_ylabel("$\\max_c |h_{\\rm spec}(c)-h_{\\rm spec}(1)|$")
ax.set_title(f"Sensitivity to a microscopic mesh defect, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
STRESS_GAP = 0.2
stress_data = mesh_stress_results[np.isclose(mesh_stress_results["gap_fraction"], STRESS_GAP)]
fig, ax = plt.subplots(figsize=(11, 7))
for current_N in sorted(stress_data["N"].unique()):
    subset = stress_data[stress_data["N"] == current_N].sort_values("alpha")
    ax.plot(
        subset["alpha"],
        subset["log_similarity_ratio"],
        marker="o",
        markersize=4,
        linewidth=1.8,
        label=f"$N={current_N}$",
    )
ax.axvline(2.0, linestyle="--", linewidth=1.5, color="black", label="critical $\\alpha=2$")
ax.axhline(0.0, linestyle=":", linewidth=1.0, color="black")
ax.set_xlabel("exponent $\\alpha$ in $\\lambda_N=N^\\alpha$")
ax.set_ylabel("$\\log\\left(S_{\\rm close}/S_{\\rm regular}\\right)$")
ax.set_title(f"Similarity advantage of compressed gap $c={STRESS_GAP}$")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
scale_data = mesh_stress_results[np.isclose(mesh_stress_results["gap_fraction"], 1.0)].copy()
fig, ax = plt.subplots(figsize=(11, 7))
for current_N in sorted(scale_data["N"].unique()):
    subset = scale_data[scale_data["N"] == current_N].sort_values("alpha")
    ax.plot(
        subset["alpha"],
        subset["lambda_delta_psi_squared"],
        marker="o",
        markersize=4,
        linewidth=1.8,
        label=f"$N={current_N}$",
    )
ax.axvline(2.0, linestyle="--", linewidth=1.5, color="black", label="critical $\\alpha=2$")
ax.axhline(1.0, linestyle=":", linewidth=1.0, color="black")
ax.set_yscale("log")
ax.set_xlabel("exponent $\\alpha$ in $\\lambda_N=N^\\alpha$")
ax.set_ylabel("$\\lambda_N\\Delta_\\psi^2$")
ax.set_title(f"Critical mesh scale, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


### Reading the mesh-robustness comparison

Each colored path corresponds to one of the six named exponents. A horizontal path indicates that the selected bandwidth is insensitive to the isolated close pair. A jump toward $h\approx0.7$ indicates mesh locking.

The theoretical expectation is that exponents below two are increasingly well resolved as $N$ grows, whereas exponents above two become increasingly dependent on individual neighboring grid points. The exact perturbation level at which a path jumps is a finite-$N$ robustness diagnostic, not a new asymptotic boundary.

This experiment remains outside the perfectly uniform-grid assumption. Its purpose is to distinguish a resolved continuous-kernel selection from a fragile nearest-neighbor selection when both happen to return approximately $h_\infty$ on the ideal grid.


## Interpretation

- The three admissible sequences approach the same sample-conditional continuous reference, although $N^{1/2}$ localizes more slowly than $N$ and $N^{3/2}$.
- With $\lambda_N=1$, the kernel never becomes globally localized. Its selector can stabilize, but not at the large-$\lambda$ criterion.
- With $\lambda_N=N^2$, the neighbor exponent remains at mesh scale instead of tending to zero. The selected bandwidth may look plausible, but it is a discretized-kernel limit rather than the target continuous local limit.
- With $\lambda_N=N^3$, neighboring similarities collapse toward zero and the matrix approaches the identity. The Perron eigenvalue becomes nearly degenerate, so an apparently accurate argmax is not evidence of the desired limit.

This is why the mesh diagnostic and final table are part of Experiment 1: the bandwidth path alone cannot verify the joint-limit assumptions.

## Scope and reproducibility

Change SEED or N_DATA only for a robustness rerun; keep them fixed within a run. The joint-limit sweep intentionally makes no claim about $n\to\infty$. The follow-up experiment below keeps $n$ fixed and freezes the numerically calibrated pair $(N,\lambda)=(1000,1000)$ while varying only the sample.

## Follow-up experiment: sampling distribution at calibrated $(N,\lambda)$

The fixed-sample experiment shows that the admissible choice $(N,\lambda_N)=(1000,1000)$ is already close to the continuous sample-conditional selector. We now freeze these numerical parameters and draw $M=1000$ independent samples of size $n=200$ from $N(0,1)$. Thus, only the data vary across repetitions.

For each sample we compute the same beta-2 spectral selector on the same geometric bandwidth grid. The resulting histogram estimates the finite-$n$ sampling distribution of $\widehat h_{1000,1000}$. For comparison, the numerical summary reports $h_{\mathbb E Q}=\arg\min_h\mathbb E[\widehat Q_{n,2}(h)]$, rather than the expectation of the random minimizer.

Every repetition uses the centralized stabilized `eigh` solver in `kde_tools.perron`. The first repetition reports its eigenpair residual and spectral gap; the compatibility matrix-vector count is zero for this direct solver.


In [ ]:
REPEATED_SAMPLE_M = 100
REPEATED_SAMPLE_N = 1000
REPEATED_SAMPLE_LAMBDA = 1000.0
REPEATED_SAMPLE_SEED = 20260903
repeated_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=H_MIN, h_max=H_MAX, beta=BETA, grid_size=REPEATED_SAMPLE_N
)
joint_spectral_cache_root = code_directory / "outputs/data/shared_spectral_sweeps"
repeated_samples_bank = np.random.default_rng(REPEATED_SAMPLE_SEED).normal(
    0.0, 1.0, size=(REPEATED_SAMPLE_M, N_DATA)
)
repeated_settings = {
    "repetitions": REPEATED_SAMPLE_M,
    "maximum_sample_size": N_DATA,
    "seed": REPEATED_SAMPLE_SEED,
    "sigma2": 1.0,
    "fft_half_width": FFT_HALF_WIDTH,
    "fft_spatial_points": FFT_SPATIAL_POINTS,
}
repeated_start_time = time.perf_counter()
repeated_spectral_result = run_l2_point(
    N_DATA,
    repeated_samples_bank,
    repeated_h_grid,
    repeated_settings,
    [REPEATED_SAMPLE_LAMBDA],
    pdf_output_directory.parents[1]
    / "data/experiment_1_joint_N_lambda_limit/shared_runner"
    / DEFAULT_FOURIER_GRID_TAG
    / f"repeated_N{REPEATED_SAMPLE_N}_M{REPEATED_SAMPLE_M}_seed{REPEATED_SAMPLE_SEED}",
    cache_root=joint_spectral_cache_root,
    diagnostics=True,
)
repeated_selected_index = repeated_spectral_result["selected_indices"][:, 0]
repeated_selected_h = repeated_h_grid[repeated_selected_index]
repeated_diagnostics = repeated_spectral_result["solver_diagnostics"][:, 0]
repeated_solver_matvecs = repeated_diagnostics[:, 2].astype(int)
solver_validation = {
    "selected index": int(repeated_selected_index[0]),
    "selected bandwidth": float(repeated_selected_h[0]),
    "relative residual": float(repeated_diagnostics[0, 0]),
    "relative spectral gap": float(repeated_diagnostics[0, 1]),
    "solver matvecs": int(repeated_solver_matvecs[0]),
}
repeated_elapsed = time.perf_counter() - repeated_start_time
repeated_sample_results = pd.DataFrame(
    {
        "repetition": np.arange(1, REPEATED_SAMPLE_M + 1),
        "selected_index": repeated_selected_index,
        "selected_h": repeated_selected_h,
        "solver_matvecs": repeated_solver_matvecs,
        "n": N_DATA,
        "M": REPEATED_SAMPLE_M,
        "N": REPEATED_SAMPLE_N,
        "lambda": REPEATED_SAMPLE_LAMBDA,
        "seed": REPEATED_SAMPLE_SEED,
        "beta": BETA,
        "h_min": H_MIN,
        "h_max": H_MAX,
        "fft_half_width": FFT_HALF_WIDTH,
        "fft_spatial_points": FFT_SPATIAL_POINTS,
        "l2_method": FOURIER_L2_METHOD_VERSION,
        "perron_solver": PERRON_SOLVER_VERSION,
    }
)
repeated_data_directory = (
    pdf_output_directory.parents[1]
    / "data"
    / "experiment_1_joint_N_lambda_limit"
    / PERRON_SOLVER_VERSION
    / FOURIER_L2_METHOD_VERSION
    / DEFAULT_FOURIER_GRID_TAG
)
repeated_data_directory.mkdir(parents=True, exist_ok=True)
repeated_results_path = (
    repeated_data_directory
    / f"repeated_sample_selections_n{N_DATA}_N{REPEATED_SAMPLE_N}_lambda{REPEATED_SAMPLE_LAMBDA:g}_M{REPEATED_SAMPLE_M}.csv"
)
repeated_grid_path = repeated_data_directory / f"bandwidth_grid_N{REPEATED_SAMPLE_N}.csv"
repeated_sample_results.to_csv(repeated_results_path, index=False)
pd.DataFrame({"grid_index": np.arange(REPEATED_SAMPLE_N), "bandwidth_h": repeated_h_grid}).to_csv(
    repeated_grid_path, index=False
)
print(f"Completed all {REPEATED_SAMPLE_M} repetitions in {repeated_elapsed:.1f} seconds.")
print(f"Saved repetition data: {repeated_results_path}")
print(f"Saved bandwidth grid: {repeated_grid_path}")
display(pd.DataFrame([solver_validation]).round(10))


In [ ]:
expected_q_minimum = minimize_bandwidth_energy(
    lambda z: expected_gaussian_qhat_log_energy(z, N_DATA), H_MIN, H_MAX, config=ENERGY_SEARCH
)
if expected_q_minimum["diagnostics"]["resolution_stable"] is False:
    raise ArithmeticError("Energy search did not stabilize; increase ENERGY_SEARCH resolution.")
h_expected_q = expected_q_minimum["global_minimum"]["h"]
sampling_summary = pd.DataFrame(
    {
        "n": [N_DATA],
        "M": [REPEATED_SAMPLE_M],
        "N": [REPEATED_SAMPLE_N],
        "lambda": [REPEATED_SAMPLE_LAMBDA],
        "mean selected h": [repeated_selected_h.mean()],
        "median selected h": [np.median(repeated_selected_h)],
        "sd selected h": [repeated_selected_h.std(ddof=1)],
        "5% quantile": [np.quantile(repeated_selected_h, 0.05)],
        "95% quantile": [np.quantile(repeated_selected_h, 0.95)],
        "argmin E[Q_hat]": [h_expected_q],
        "mean solver matvecs": [repeated_solver_matvecs.mean()],
        "elapsed seconds": [repeated_elapsed],
    }
)
display(sampling_summary.round(6))
sampling_summary_path = (
    repeated_data_directory
    / f"repeated_sample_summary_n{N_DATA}_N{REPEATED_SAMPLE_N}_lambda{REPEATED_SAMPLE_LAMBDA:g}_M{REPEATED_SAMPLE_M}.csv"
)
sampling_summary.to_csv(sampling_summary_path, index=False)
print(f"Saved summary data: {sampling_summary_path}")
repeated_selection_counts = np.bincount(repeated_selected_index, minlength=REPEATED_SAMPLE_N)
repeated_histogram = plot_bandwidth_distributions(
    h_grid=repeated_h_grid,
    distributions={"Spectral selections": repeated_selection_counts},
    colors={"Spectral selections": "#005A9C"},
    opacity=1.0,
    width=950,
    height=450,
)
repeated_histogram.update_traces(marker_line_width=0)
repeated_histogram.update_yaxes(tickformat=".1%")
repeated_histogram.update_layout(title="Sampling distribution of the calibrated spectral selector")
repeated_histogram.show()


### Interpretation of the repeated-sample experiment

Increasing $M$ estimates the sampling distribution more accurately; it does not alter the selector. Because $n=200$ remains fixed, sampling variability persists even though the numerical grid and localization parameters have already been calibrated. The comparison with $h_{\mathbb E Q}$ tests whether the distribution is approximately centered near the minimizer of the expected criterion. Equality is not automatic, since expectation and minimization need not commute.


## Sampling distributions for increasing sample size

We keep the calibrated numerical parameters $(N,\lambda)=(1000,1000)$ and the number of repetitions $M=1000$ fixed, while increasing the sample size through $n=100,200,\ldots,1000$. The samples are coupled across sample sizes: repetition $m$ uses the first $n$ observations of one common sample of length 1000. Each selection table is saved immediately after its sample size has finished.


In [ ]:
SAMPLE_SIZE_SWEEP_VALUES = np.arange(100, 1001, 100, dtype=int)
SAMPLE_SIZE_SWEEP_M = 1000
SAMPLE_SIZE_SWEEP_N = 1000
SAMPLE_SIZE_SWEEP_LAMBDA = 1000.0
SAMPLE_SIZE_SWEEP_SEED = REPEATED_SAMPLE_SEED + 1
sample_size_sweep_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=H_MIN, h_max=H_MAX, beta=BETA, grid_size=SAMPLE_SIZE_SWEEP_N
)
sample_size_sweep_rng = np.random.default_rng(SAMPLE_SIZE_SWEEP_SEED)
sample_size_sweep_full_samples = sample_size_sweep_rng.normal(
    loc=0.0, scale=1.0, size=(SAMPLE_SIZE_SWEEP_M, int(SAMPLE_SIZE_SWEEP_VALUES.max()))
)
sample_size_sweep_data_directory = (
    pdf_output_directory.parents[1]
    / "data"
    / "experiment_1_joint_N_lambda_limit"
    / PERRON_SOLVER_VERSION
    / FOURIER_L2_METHOD_VERSION
    / DEFAULT_FOURIER_GRID_TAG
    / "sample_size_sweep"
)
sample_size_sweep_data_directory.mkdir(parents=True, exist_ok=True)
sample_size_sweep_results = {}
sample_size_sweep_summary_rows = []
for current_n in SAMPLE_SIZE_SWEEP_VALUES:
    current_n = int(current_n)
    current_start_time = time.perf_counter()
    current_settings = {
        "repetitions": SAMPLE_SIZE_SWEEP_M,
        "maximum_sample_size": int(SAMPLE_SIZE_SWEEP_VALUES.max()),
        "seed": SAMPLE_SIZE_SWEEP_SEED,
        "sigma2": 1.0,
        "fft_half_width": FFT_HALF_WIDTH,
        "fft_spatial_points": FFT_SPATIAL_POINTS,
    }
    current_spectral_result = run_l2_point(
        current_n,
        sample_size_sweep_full_samples,
        sample_size_sweep_h_grid,
        current_settings,
        [SAMPLE_SIZE_SWEEP_LAMBDA],
        sample_size_sweep_data_directory
        / "shared_runner"
        / f"N{SAMPLE_SIZE_SWEEP_N}_M{SAMPLE_SIZE_SWEEP_M}_seed{SAMPLE_SIZE_SWEEP_SEED}",
        cache_root=joint_spectral_cache_root,
        diagnostics=True,
    )
    selected_indices = current_spectral_result["selected_indices"][:, 0]
    solver_matvecs = current_spectral_result["solver_diagnostics"][:, 0, 2].astype(int)
    elapsed_seconds = time.perf_counter() - current_start_time
    selected_bandwidths = sample_size_sweep_h_grid[selected_indices]
    selection_counts = np.bincount(selected_indices, minlength=SAMPLE_SIZE_SWEEP_N)
    current_results = pd.DataFrame(
        {
            "repetition": np.arange(1, SAMPLE_SIZE_SWEEP_M + 1),
            "selected_index": selected_indices,
            "selected_h": selected_bandwidths,
            "solver_matvecs": solver_matvecs,
            "n": current_n,
            "M": SAMPLE_SIZE_SWEEP_M,
            "N": SAMPLE_SIZE_SWEEP_N,
            "lambda": SAMPLE_SIZE_SWEEP_LAMBDA,
            "seed": SAMPLE_SIZE_SWEEP_SEED,
            "beta": BETA,
            "h_min": H_MIN,
            "h_max": H_MAX,
            "fft_half_width": FFT_HALF_WIDTH,
            "fft_spatial_points": FFT_SPATIAL_POINTS,
            "l2_method": FOURIER_L2_METHOD_VERSION,
            "perron_solver": PERRON_SOLVER_VERSION,
        }
    )
    current_data_path = (
        sample_size_sweep_data_directory
        / f"selections_n{current_n}_N{SAMPLE_SIZE_SWEEP_N}_lambda{SAMPLE_SIZE_SWEEP_LAMBDA:g}_M{SAMPLE_SIZE_SWEEP_M}.csv"
    )
    current_results.to_csv(current_data_path, index=False)
    sample_size_sweep_results[current_n] = {
        "selected_indices": selected_indices.copy(),
        "selected_bandwidths": selected_bandwidths.copy(),
        "counts": selection_counts,
        "data_path": current_data_path,
    }
    sample_size_sweep_summary_rows.append(
        {
            "n": current_n,
            "M": SAMPLE_SIZE_SWEEP_M,
            "N": SAMPLE_SIZE_SWEEP_N,
            "lambda": SAMPLE_SIZE_SWEEP_LAMBDA,
            "mean selected h": selected_bandwidths.mean(),
            "median selected h": np.median(selected_bandwidths),
            "sd selected h": selected_bandwidths.std(ddof=1),
            "5% quantile": np.quantile(selected_bandwidths, 0.05),
            "95% quantile": np.quantile(selected_bandwidths, 0.95),
            "mean solver matvecs": solver_matvecs.mean(),
            "elapsed seconds": elapsed_seconds,
        }
    )
    print(f"Saved n={current_n} data: {current_data_path}")
sample_size_sweep_summary = pd.DataFrame(sample_size_sweep_summary_rows)
sample_size_sweep_summary_path = sample_size_sweep_data_directory / "summary.csv"
sample_size_sweep_summary.to_csv(sample_size_sweep_summary_path, index=False)
display(sample_size_sweep_summary.round(6))
print(f"Saved sample-size summary: {sample_size_sweep_summary_path}")


In [ ]:
sample_size_sweep_figures = {}
for current_n in SAMPLE_SIZE_SWEEP_VALUES:
    current_n = int(current_n)
    current_figure = plot_bandwidth_distributions(
        h_grid=sample_size_sweep_h_grid,
        distributions={"Spectral selections": sample_size_sweep_results[current_n]["counts"]},
        colors={"Spectral selections": "#005A9C"},
        opacity=1.0,
        width=950,
        height=450,
    )
    current_figure.update_traces(marker_line_width=0)
    current_figure.update_yaxes(tickformat=".1%")
    current_figure.update_layout(
        title=f"Sampling distribution of the spectral selector (n={current_n}, N=1000, lambda=1000)"
    )
    sample_size_sweep_figures[current_n] = current_figure
    current_figure.show()


In [ ]:
fixed_error_results = custom_beta_results.copy()
fixed_error_results["alpha"] = np.log(fixed_error_results["lambda"]) / np.log(
    fixed_error_results["N"]
)
fixed_error_results = (
    fixed_error_results[np.isclose(fixed_error_results["alpha"], 1.0, rtol=0.0, atol=1e-10)]
    .sort_values("N")
    .copy()
)
psi_reference = psi_beta(custom_beta_h_infinity, CUSTOM_BETA)
fixed_error_results["psi_error"] = [
    abs(psi_beta(h, CUSTOM_BETA) - psi_reference) for h in fixed_error_results["selected_h"]
]
psi_interval_length = abs(psi_beta(H_MAX, CUSTOM_BETA) - psi_beta(H_MIN, CUSTOM_BETA))
fixed_error_results["delta_psi"] = psi_interval_length / (fixed_error_results["N"] - 1)
fixed_error_results["error_in_grid_cells"] = (
    fixed_error_results["psi_error"] / fixed_error_results["delta_psi"]
)
fig, ax = plt.subplots(figsize=(9.5, 5.8))
ax.plot(
    fixed_error_results["N"],
    np.maximum(fixed_error_results["psi_error"].to_numpy(), 1e-16),
    marker="o",
    linewidth=2.2,
    label="$|\\psi_\\beta(h_{\\rm Perron})-\\psi_\\beta(\\hat h_Q)|$",
)
ax.plot(
    fixed_error_results["N"],
    fixed_error_results["delta_psi"],
    linestyle="--",
    linewidth=1.8,
    label="grid spacing $\\Delta_\\psi$",
)
ax.set(
    xscale="log",
    yscale="log",
    xlabel="number of bandwidths $N$",
    ylabel="error in $\\psi_\\beta$ coordinate",
    title=f"Fixed-sample Perron convergence to $\\arg\\min \\widehat Q_{{n,\\beta}}$, $\\beta={CUSTOM_BETA:g}$",
)
ax.legend()
fig.tight_layout()
plt.show()
display(
    fixed_error_results[
        [
            "N",
            "lambda",
            "selected_h",
            "continuous_reference",
            "psi_error",
            "delta_psi",
            "error_in_grid_cells",
        ]
    ].round(8)
)
MECHANISM_N = 1000
MECHANISM_ALPHA = 1.0
MECHANISM_LAMBDA = float(MECHANISM_N**MECHANISM_ALPHA)
mechanism_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=H_MIN, h_max=H_MAX, beta=CUSTOM_BETA, grid_size=MECHANISM_N
)
mechanism_psi_grid = np.array([psi_beta(h, CUSTOM_BETA) for h in mechanism_h_grid])
mechanism_delta_psi = float(mechanism_psi_grid[1] - mechanism_psi_grid[0])
mechanism_divergence = beta2_divergence_matrix_fourier(
    samples, mechanism_h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
)
mechanism_divergence = np.maximum(mechanism_divergence, 0.0)
mechanism_similarity = np.exp(-MECHANISM_LAMBDA * mechanism_divergence)
mechanism_row_integral = mechanism_delta_psi * mechanism_similarity.sum(axis=1)
mechanism_Q = np.array(
    [local_custom_beta_metric(h, squared_differences, beta=CUSTOM_BETA) for h in mechanism_h_grid]
)
mechanism_gaussian_prediction = np.sqrt(np.pi / (MECHANISM_LAMBDA * mechanism_Q))
mechanism_perron_vector = perron_vector_from_divergence(mechanism_divergence, MECHANISM_LAMBDA)
mechanism_h_Q = float(mechanism_h_grid[np.argmin(mechanism_Q)])
mechanism_h_row = float(mechanism_h_grid[np.argmax(mechanism_row_integral)])
mechanism_h_perron = float(mechanism_h_grid[perron_peak_index(mechanism_perron_vector)])
fig, ax = plt.subplots(figsize=(9.5, 5.8))
ax.plot(
    mechanism_h_grid, mechanism_row_integral, linewidth=2.2, label="$\\Delta_\\psi\\sum_j S_{ij}$"
)
ax.plot(
    mechanism_h_grid,
    mechanism_gaussian_prediction,
    linestyle="--",
    linewidth=2.0,
    label="$\\sqrt{\\pi/(\\lambda Q_i)}$",
)
ax.axvline(
    mechanism_h_Q, linestyle=":", linewidth=1.8, label=f"$\\arg\\min Q_i={mechanism_h_Q:.4f}$"
)
ax.axvline(
    mechanism_h_row, linestyle="-.", linewidth=1.6, label=f"$\\arg\\max R_i={mechanism_h_row:.4f}$"
)
ax.axvline(
    mechanism_h_perron,
    linestyle=(0, (5, 2)),
    linewidth=1.6,
    label=f"Perron peak $={mechanism_h_perron:.4f}$",
)
ax.set(
    xscale="log",
    xlabel="bandwidth $h$",
    ylabel="localized row mass",
    title=f"Row-sum mechanism, $N={MECHANISM_N}$, $\\lambda=N^{{{MECHANISM_ALPHA:g}}}$, $\\beta={CUSTOM_BETA:g}$",
)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()
mechanism_summary = pd.DataFrame(
    {
        "quantity": ["argmin local Q", "argmax row sum", "Perron peak"],
        "bandwidth": [mechanism_h_Q, mechanism_h_row, mechanism_h_perron],
    }
)
display(mechanism_summary.round(8))
ASYMPTOTIC_BETA = float(CUSTOM_BETA)
if not -2.0 < ASYMPTOTIC_BETA < 3.0:
    raise ValueError("The interior asymptotic minimizer requires -2 < beta < 3.")
ASYMPTOTIC_M = 200
ASYMPTOTIC_N_VALUES = np.array([100, 200, 400, 800, 1600, 3200], dtype=int)
ASYMPTOTIC_SEED = 20260906
ASYMPTOTIC_T_MIN = 0.35
ASYMPTOTIC_T_MAX = 2.2
ASYMPTOTIC_GRID_BASE = 250
ASYMPTOTIC_GRID_GAMMA = 0.25
ASYMPTOTIC_ALPHA = 1.0
ASYMPTOTIC_C_BETA = asymptotic_c_beta(ASYMPTOTIC_BETA)
print("Theoretical asymptotic constant:", ASYMPTOTIC_C_BETA)
if not ASYMPTOTIC_T_MIN < ASYMPTOTIC_C_BETA < ASYMPTOTIC_T_MAX:
    raise ValueError("The theoretical c_beta must lie inside the chosen scaled t-window.")
asymptotic_rng = np.random.default_rng(ASYMPTOTIC_SEED)
asymptotic_full_samples = asymptotic_rng.normal(
    loc=0.0, scale=1.0, size=(ASYMPTOTIC_M, int(ASYMPTOTIC_N_VALUES.max()))
)
asymptotic_rows = []
local_metric_validation = None
asymptotic_start_time = time.perf_counter()
for current_n in ASYMPTOTIC_N_VALUES:
    current_n = int(current_n)
    current_N = asymptotic_grid_size(
        current_n,
        base_size=ASYMPTOTIC_GRID_BASE,
        gamma=ASYMPTOTIC_GRID_GAMMA,
        sample_sizes=ASYMPTOTIC_N_VALUES,
    )
    current_t_grid = bandwidth_grid_linear_in_psi_beta(
        h_min=ASYMPTOTIC_T_MIN, h_max=ASYMPTOTIC_T_MAX, beta=ASYMPTOTIC_BETA, grid_size=current_N
    )
    current_scale = current_n ** (-0.2)
    current_h_grid = current_scale * current_t_grid
    current_psi_grid = np.array([psi_beta(h, ASYMPTOTIC_BETA) for h in current_h_grid])
    current_delta_psi = float(current_psi_grid[1] - current_psi_grid[0])
    current_lambda = float(current_n**0.8 * current_N**ASYMPTOTIC_ALPHA)
    current_effective_lambda = float(current_lambda * current_n ** (-0.8))
    current_h_expected = exact_expected_q_minimizer(
        current_n, ASYMPTOTIC_BETA, ASYMPTOTIC_T_MIN, ASYMPTOTIC_T_MAX, search=ENERGY_SEARCH
    )
    current_t_expected = float(current_n**0.2 * current_h_expected)
    current_start_time = time.perf_counter()
    for repetition in range(ASYMPTOTIC_M):
        current_samples = asymptotic_full_samples[repetition, :current_n]
        current_divergence = beta2_divergence_matrix_fourier(
            current_samples,
            current_h_grid,
            half_width=FFT_HALF_WIDTH,
            spatial_points=FFT_SPATIAL_POINTS,
        )
        current_divergence = np.maximum(current_divergence, 0.0)
        np.fill_diagonal(current_divergence, 0.0)
        perron_result = perron_eigenpair_from_divergence(current_divergence, current_lambda)
        current_perron_vector = perron_result.vector
        matvec_count = perron_result.matvecs
        current_perron_index = int(perron_peak_index(current_perron_vector))
        current_h_perron = float(current_h_grid[current_perron_index])
        current_q_local = richardson_local_metric_from_divergence(
            current_divergence, current_delta_psi
        )
        finite_q_indices = np.flatnonzero(np.isfinite(current_q_local))
        if finite_q_indices.size == 0:
            raise RuntimeError("No finite positive local-metric estimates.")
        current_q_index = int(finite_q_indices[np.argmin(current_q_local[finite_q_indices])])
        current_h_q = float(current_h_grid[current_q_index])
        current_t_perron = float(current_n**0.2 * current_h_perron)
        current_t_q = float(current_n**0.2 * current_h_q)
        asymptotic_rows.append(
            {
                "n": current_n,
                "repetition": repetition + 1,
                "N": current_N,
                "lambda": current_lambda,
                "effective_lambda": current_effective_lambda,
                "effective_lambda_over_N2": current_effective_lambda / current_N**2,
                "beta": ASYMPTOTIC_BETA,
                "h_perron": current_h_perron,
                "h_q": current_h_q,
                "h_expected_q": current_h_expected,
                "t_perron": current_t_perron,
                "t_q": current_t_q,
                "t_expected_q": current_t_expected,
                "c_beta": ASYMPTOTIC_C_BETA,
                "scaled_spectral_gap": abs(current_t_perron - current_t_q),
                "solver_matvecs": matvec_count,
            }
        )
        if (repetition + 1) % 25 == 0 or repetition + 1 == ASYMPTOTIC_M:
            print(f"n={current_n}, N={current_N}: {repetition + 1}/{ASYMPTOTIC_M} repetitions")
    current_elapsed = time.perf_counter() - current_start_time
    print(
        f"Finished n={current_n}, N={current_N}, lambda={current_lambda:.4g} in {current_elapsed:.1f} s."
    )
asymptotic_results = pd.DataFrame(asymptotic_rows)
asymptotic_elapsed = time.perf_counter() - asymptotic_start_time
print(f"Completed full asymptotic experiment in {asymptotic_elapsed:.1f} s.")
asymptotic_summary = asymptotic_results.groupby("n", as_index=False).agg(
    N=("N", "first"),
    lambda_value=("lambda", "first"),
    effective_lambda=("effective_lambda", "first"),
    effective_lambda_over_N2=("effective_lambda_over_N2", "first"),
    mean_t_perron=("t_perron", "mean"),
    median_t_perron=("t_perron", "median"),
    q10_t_perron=("t_perron", lambda x: np.quantile(x, 0.1)),
    q90_t_perron=("t_perron", lambda x: np.quantile(x, 0.9)),
    mean_t_q=("t_q", "mean"),
    median_t_q=("t_q", "median"),
    q10_t_q=("t_q", lambda x: np.quantile(x, 0.1)),
    q90_t_q=("t_q", lambda x: np.quantile(x, 0.9)),
    t_expected_q=("t_expected_q", "first"),
    c_beta=("c_beta", "first"),
    mean_scaled_spectral_gap=("scaled_spectral_gap", "mean"),
    median_scaled_spectral_gap=("scaled_spectral_gap", "median"),
)
display(asymptotic_summary.round(8))
fig, ax = plt.subplots(figsize=(9.8, 6.0))
ax.plot(
    asymptotic_summary["n"],
    asymptotic_summary["mean_t_perron"],
    marker="o",
    linewidth=2.2,
    label="MC mean of $n^{1/5}h_{\\rm Perron}$",
)
ax.plot(
    asymptotic_summary["n"],
    asymptotic_summary["mean_t_q"],
    marker="s",
    linewidth=2.0,
    label="MC mean of $n^{1/5}\\arg\\min\\widehat Q$",
)
ax.plot(
    asymptotic_summary["n"],
    asymptotic_summary["t_expected_q"],
    marker="^",
    linestyle="--",
    linewidth=1.9,
    label="$n^{1/5}\\arg\\min E[\\widehat Q]$",
)
ax.axhline(
    ASYMPTOTIC_C_BETA, linestyle=":", linewidth=2.0, label=f"$c_\\beta={ASYMPTOTIC_C_BETA:.4f}$"
)
ax.set(
    xscale="log",
    xlabel="sample size $n$",
    ylabel="rescaled bandwidth $n^{1/5}h$",
    title=f"Full Perron-to-asymptotic bridge, $\\beta={ASYMPTOTIC_BETA:g}$",
)
ax.legend()
fig.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(9.5, 5.8))
ax.plot(
    asymptotic_summary["n"],
    asymptotic_summary["mean_scaled_spectral_gap"],
    marker="o",
    linewidth=2.2,
    label="Monte Carlo mean",
)
ax.plot(
    asymptotic_summary["n"],
    asymptotic_summary["median_scaled_spectral_gap"],
    marker="s",
    linestyle="--",
    linewidth=1.8,
    label="Monte Carlo median",
)
ax.axhline(0.0, linestyle=":", linewidth=1.5)
ax.set(
    xscale="log",
    xlabel="sample size $n$",
    ylabel="$n^{1/5}|h_{\\rm Perron}-\\hat h_Q|$",
    title="Spectral error on the statistical bandwidth scale",
)
ax.legend()
fig.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(9.5, 5.8))
n_values_plot = asymptotic_summary["n"].to_numpy(dtype=float)
ax.fill_between(
    n_values_plot,
    asymptotic_summary["q10_t_perron"].to_numpy(dtype=float),
    asymptotic_summary["q90_t_perron"].to_numpy(dtype=float),
    alpha=0.2,
    label="10%-90% Monte Carlo band",
)
ax.plot(
    n_values_plot,
    asymptotic_summary["median_t_perron"],
    marker="o",
    linewidth=2.2,
    label="median $n^{1/5}h_{\\rm Perron}$",
)
ax.axhline(
    ASYMPTOTIC_C_BETA, linestyle=":", linewidth=2.0, label=f"$c_\\beta={ASYMPTOTIC_C_BETA:.4f}$"
)
ax.set(
    xscale="log",
    xlabel="sample size $n$",
    ylabel="$n^{1/5}h_{\\rm Perron}$",
    title="Sampling concentration of the rescaled Perron peak",
)
ax.legend()
fig.tight_layout()
plt.show()


# Additional grid parameters: r=1 and r=13/14


# Figure 3.6 grid experiments: $r=1$ and $r=13/14$

Choose `GRID_R = "1"` or `"13/14"` in the setup cell, then run all cells in order.
The same runner calculates both grids and keeps their results separate. The
filename is retained for existing links.

The fixed sample, interval, grid sizes, lambda regimes, and numerical checks are
shared. Both grid definitions are retained below.

Sources: [thesis grid definition, Eq. `eq:def_r_linear`](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex),
[original experiment](experiment_1_joint_N_lambda_limit.ipynb), and
[shared runner](../experiments/joint_limits/run_experiment.py).


## Setup and grid choice

The exporter is enabled before plotting. The r=1 experiment retains
`outputs/pdfs/linear_grid_joint_n_lambda_limit/`; r=13/14 retains
`outputs/pdfs/r13over14_joint_n_lambda_limit/`. Each choice also retains its
own data and image directories. Rerun setup when changing the choice.


In [ ]:
GRID_R = "1"
experiment_name = grid_experiment.configure_grid(GRID_R)
pdf_directory = pdf_output_directory


## Run and validate the selected grid

The runner compares both endpoints and all detected interior minima of the
matching $\widehat Q_r$ criterion on $[0.01,2]$. It retains the exact-Gaussian
cross-check, doubled frequency-resolution check, and eigenpair diagnostics.
Numerical settings are recorded with each experiment's checkpoints.

Sources: [thesis Proposition `prop:main_thm`](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex)
and [runner and validation](../experiments/joint_limits/run_experiment.py).


In [ ]:
with threadpool_limits(limits=2):
    results, reference, metric_h, metric_q = grid_experiment.run_experiment()
print("Continuous reference on [0.01, 2]:", reference)
display(results.pivot(index="N", columns="alpha", values="selected_h"))


In [ ]:
image_path = grid_experiment.plot_results(results, reference)
print("PNG:", image_path)
print("Automatic PDFs:", pdf_directory)


## Grid definition: r=1

Repeat the current Figure 3.6 fixed-sample joint $(N,\lambda_N)$ experiment, changing only the bandwidth grid to

$$h_i=0.01+\frac{i-1}{N-1}(2-0.01),\qquad i=1,\ldots,N.$$

The sample is the same 200 standard-normal draws from NumPy's `default_rng(20260823)`. The interval remains $[0.01,2]$, $N=100,200,\ldots,3000$, and $\lambda_N\in\{1,N^{1/2},N,N^{3/2},N^2,N^3\}$. The grid helper calls this `beta=0`, corresponding to thesis $r=1$ through $\beta=2(1-r)$; the divergence remains one half of the squared $L^2$ distance.

Sources: [current Figure 3.6 and grid definition, Eq. `eq:def_r_linear`](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [original experiment, initial setup and sweep](experiment_1_joint_N_lambda_limit.ipynb); [grid helper](../kde_tools/bandwidths.py).

## Grid definition: r=13/14

Repeat the fixed-sample joint $(N,\lambda_N)$ experiment with

$$h_i=\left[0.01^{13/14}+\frac{i-1}{N-1}\left(2^{13/14}-0.01^{13/14}\right)\right]^{14/13}.$$

The sample and all other settings match the preceding linear-grid run: 200 standard-normal draws from `default_rng(20260823)`, bandwidth interval $[0.01,2]$, $N=100,200,\ldots,3000$, and $\lambda_N\in\{1,N^{1/2},N,N^{3/2},N^2,N^3\}$. The grid helper uses $\beta=2(1-r)=1/7$; the divergence remains one half of the squared $L^2$ distance.

Sources: [thesis grid definition, Eq. `eq:def_r_linear`](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [original Figure 3.6 workflow](experiment_1_joint_N_lambda_limit.ipynb); [grid helper](../kde_tools/bandwidths.py); [grid choice above](#setup-and-grid-choice).

## Current results and files

The run cell displays the selected grid's current table and interval reference.
The plot cell uses those same results. Earlier hardcoded result tables and
links to absent pre-unification files have been removed from this notebook.
The runner records the current results, reference, and validation under the
chosen experiment's `outputs/data/<experiment>/unified_<settings-id>/` directory.
Existing files from earlier runs remain unchanged.
